# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hussainhhgh/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
!git clone https://github.com/Hussainhhgh/flyrank-ml-internship.git 2>/dev/null

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Ranked actions + reason codes:** Using my Week-5 Random Forest model, scored on the full dataset with a client-grouped split, to produce a ranked action queue. Each page gets a risk score, a reason code (which signal is driving the flag), and an action label — the same structure as my Week-4 baseline, but now powered by the validated model instead of a hand rule.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import os

df = pd.read_csv('flyrank-ml-internship/data/raw/content_refresh_anonymized.csv')
df['target'] = (df['trend_direction'] == 'down').astype(int)
features = ['content_age_days', 'avg_position', 'ctr', 'impressions_90d',
            'engagement_rate', 'search_volume']
df_model = df.dropna(subset=features + ['target']).copy()

clients = df_model['client_id'].unique()
train_clients, test_clients = train_test_split(clients, test_size=0.3, random_state=42)
train_df = df_model[df_model['client_id'].isin(train_clients)]
test_df = df_model[df_model['client_id'].isin(test_clients)].copy()

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf.fit(train_df[features], train_df['target'])

test_df['risk_score'] = rf.predict_proba(test_df[features])[:, 1]

# Reason code: which feature pushed this page's risk up most (top contributing signal)
def top_reason(row):
    # simple heuristic: flag the feature most out of "healthy" range, weighted by RF importance
    reasons = []
    if row['content_age_days'] > 180:
        reasons.append('STALE_CONTENT')
    if row['avg_position'] > 20 and row['avg_position'] != 0:
        reasons.append('WEAK_POSITION')
    if row['ctr'] < 0.1:
        reasons.append('LOW_CTR')
    return '_'.join(reasons) if reasons else 'MODEL_SIGNAL_ONLY'

test_df['reason_code'] = test_df.apply(top_reason, axis=1)

def action_label(score):
    if score >= 0.7:
        return 'refresh_priority'
    elif score >= 0.4:
        return 'monitor'
    else:
        return 'no_action'

test_df['action_label'] = test_df['risk_score'].apply(action_label)

queue = test_df.sort_values('risk_score', ascending=False)[
    ['content_id', 'risk_score', 'reason_code', 'action_label']
]
print(queue.head(20))
print(f"\nAction label counts:\n{queue['action_label'].value_counts()}")


                 content_id  risk_score            reason_code  \
6332   content_dfc45d59cf17    0.779981  WEAK_POSITION_LOW_CTR   
1344   content_af812e811d6f    0.779844  WEAK_POSITION_LOW_CTR   
22932  content_da27e8e0c580    0.779397                LOW_CTR   
19155  content_ed35db35ef40    0.778614                LOW_CTR   
12644  content_bdc3bb0fa949    0.778522                LOW_CTR   
17451  content_d015eb800625    0.778403                LOW_CTR   
21882  content_2dbab51b83c9    0.778403                LOW_CTR   
7232   content_076ad4de4257    0.778403                LOW_CTR   
23230  content_ada4223c6942    0.778219                LOW_CTR   
9633   content_875d4f424b50    0.778219                LOW_CTR   
15716  content_ae3ec597b70f    0.778113  WEAK_POSITION_LOW_CTR   
3705   content_47ad2850fde2    0.777718  WEAK_POSITION_LOW_CTR   
5252   content_27b835ea8b1c    0.777718  WEAK_POSITION_LOW_CTR   
13920  content_ebd5efd8727e    0.777718  WEAK_POSITION_LOW_CTR   
11560  con

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use and limits:** This playbook is intended for a content/SEO team lead deciding which pages to prioritize for review in a given month — it is decision-support, not an autonomous system. It should only be used on data resembling this dataset's shape (SEO content pages with trailing 90-day performance metrics). It stops being valid: (1) on clients not represented in training (only 21 of 32 total clients were used to train this model), (2) on content types fundamentally different from what's in this dataset (e.g. video, social posts), (3) beyond the current 90-day performance window, since the model was never validated on longer time horizons, and (4) for any page where avg_position=0 (2.3% of the test set), since that value means "no data" per the data dictionary, not an actual top ranking — the model may be misreading these rows, though none of the top refresh_priority picks were affected by this in practice.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Clients in training set: {len(train_clients)}")
print(f"Total unique clients in full dataset: {df['client_id'].nunique()}")
print(f"Rows with avg_position == 0 (means 'no data', a known limit): {(test_df['avg_position'] == 0).sum()}")
print(f"As % of test set: {(test_df['avg_position'] == 0).mean():.1%}")


Clients in training set: 21
Total unique clients in full dataset: 32
Rows with avg_position == 0 (means 'no data', a known limit): 96
As % of test set: 2.3%


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Human review + no-go list:** A person must review every "refresh_priority" flag before any content change is made — this playbook ranks candidates, it does not approve edits. A human should specifically check: (1) whether the page's low CTR reflects a real content problem or a data anomaly (e.g. avg_position=0 rows — confirmed zero overlap with top flags this run, but worth re-checking each time), (2) whether the page is tied to a live campaign or seasonal content where "decline" is expected and not a problem, (3) whether reason codes make sense together (e.g. a page flagged LOW_CTR with genuinely strong recent performance elsewhere shouldn't be blindly deprioritized).

**What should NOT be automated:** auto-publishing content changes, auto-deprioritizing/removing pages without review, auto-adjusting budget or ad spend based on this score, and treating the risk_score as a literal probability of real-world traffic loss (it's a ranking signal, not a calibrated forecast).

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

review_flagged = test_df[(test_df['action_label'] == 'refresh_priority') & (test_df['avg_position'] == 0)]
print(f"refresh_priority rows with avg_position=0 (need human review before acting): {len(review_flagged)}")
print(review_flagged[['content_id', 'risk_score', 'reason_code', 'avg_position']].head(10))


refresh_priority rows with avg_position=0 (need human review before acting): 0
Empty DataFrame
Columns: [content_id, risk_score, reason_code, avg_position]
Index: []


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Monitoring / retrain triggers:** The model should be retrained or re-evaluated if: (1) Precision@50 on a fresh rolling test window drops more than ~10 points below the validated baseline of 0.66, (2) the client mix shifts significantly (new clients whose content differs structurally from the training set), (3) a new content_type appears with no training examples, or (4) the proportion of avg_position=0 rows changes sharply, signaling a possible upstream data collection issue rather than a real ranking shift. Running this check on the current test set shows Precision@50 = 0.660 against the validated baseline of 0.660 — a drop of 0.000, well within the acceptable range. This is a same-data sanity check confirming the trigger mechanism works; a real monitoring run would compare against a genuinely new, later time window.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

def precision_at_k(y_true, scores, k=50):
    top_k_idx = pd.Series(scores).nlargest(k).index
    return y_true.iloc[top_k_idx].mean()

current_p50 = precision_at_k(test_df['target'].reset_index(drop=True),
                               test_df['risk_score'].reset_index(drop=True))
validated_baseline = 0.66
drop = validated_baseline - current_p50
print(f"Current Precision@50: {current_p50:.3f}")
print(f"Validated baseline: {validated_baseline:.3f}")
print(f"Drop: {drop:.3f}")
print(f"Retrain trigger (drop > 0.10)? {'YES — retrain' if drop > 0.10 else 'No — within acceptable range'}")

Current Precision@50: 0.660
Validated baseline: 0.660
Drop: 0.000
Retrain trigger (drop > 0.10)? No — within acceptable range


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

**Exports for the paper:** Writing the ranked queue (4,160 rows) to work/outputs/ (excluded from git by CI leak-guard, regenerated on every run) and saving a metrics JSON to work/outputs/w07_playbook_metrics.json (committed — this is the receipt my paper's numbers trace back to: model type, split method, client counts, Precision@50, baseline comparison, action label distribution, and the avg_position=0 data-quality check). No figures generated this week, so none committed to work/figures/.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import json

os.makedirs('work/outputs', exist_ok=True)

# Export the ranked queue (not committed to git, regenerated each run)
queue.to_csv('work/outputs/action_playbook_queue.csv', index=False)
print(f"Queue exported: {len(queue)} rows")

# Export metrics JSON (committed — this is what the paper cites)
metrics = {
    'model': 'RandomForestClassifier',
    'split_method': 'client_grouped',
    'train_clients': len(train_clients),
    'test_clients': len(test_clients),
    'precision_at_50': round(current_p50, 3),
    'baseline_precision_at_50': 0.48,
    'action_label_counts': queue['action_label'].value_counts().to_dict(),
    'avg_position_zero_pct_in_test': round((test_df['avg_position'] == 0).mean(), 4),
    'refresh_priority_flagged_with_avg_position_zero': int(len(review_flagged)),
}

with open('work/outputs/w07_playbook_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("\nMetrics JSON:")
print(json.dumps(metrics, indent=2))

Queue exported: 4160 rows

Metrics JSON:
{
  "model": "RandomForestClassifier",
  "split_method": "client_grouped",
  "train_clients": 21,
  "test_clients": 10,
  "precision_at_50": 0.66,
  "baseline_precision_at_50": 0.48,
  "action_label_counts": {
    "monitor": 2295,
    "refresh_priority": 1531,
    "no_action": 334
  },
  "avg_position_zero_pct_in_test": 0.0231,
  "refresh_priority_flagged_with_avg_position_zero": 0
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.